In [2]:
import os
import math
import copy
from itertools import zip_longest

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch import optim

In [3]:
def set_random_seed(state=1):
    gens = (np.random.seed, torch.manual_seed, torch.cuda.manual_seed)
    for set_state in gens:
        set_state(state)

In [4]:
RANDOM_STATE = 42
set_random_seed(RANDOM_STATE)

In [10]:
# !wget -nc $DATASET_LINK
$fullPath = "C:\Users\Admin\Documents\studies\Coding\Mastering pytorch\Chapter 18"
Invoke-WebRequest -Uri https://files.grouplens.org/datasets/movielens/ml-latest-small.zip -OutFile "$fullPath\ml-latest-small.zip
Expand-Archive -Path "$fullPath\ml-latest-small.zip" -DestinationPath $fullPath

'unzip' is not recognized as an internal or external command,
operable program or batch file.


In [15]:
def read_data(path):
    files = {}
    for filename in os.listdir(path):
        stem, suffix = os.path.splitext(filename)
        file_path = os.path.join(path, filename)
        print(filename)
        if suffix == '.csv':
            files[stem] = pd.read_csv(file_path)
        elif suffix == '.dat':
            if stem == 'ratings':
                columns = ['userId', 'movieId', 'rating', 'timestamp']
            else:
                columns = ['movieId', 'title', 'genres']
            data = pd.read_csv(file_path, sep='::', names=columns, engine='python')
            files[stem] = data
    return files['ratings'], files['movies']

ratings, movies = read_data("./ml-latest-small")

links.csv
movies.csv
ratings.csv
README.txt
tags.csv


In [20]:
minmax = ratings.rating.min(), ratings.rating.max()
minmax

(0.5, 5.0)

In [22]:
ratings = ratings.merge(movies[['movieId', 'title']], on='movieId')

In [25]:
def tabular_preview(ratings, n=15):

    user_groups = ratings.groupby('userId')['rating'].count()
    top_users = user_groups.sort_values(ascending=False)[:n]

    movie_groups = ratings.groupby('movieId')['rating'].count()
    top_movies = movie_groups.sort_values(ascending=False)[:n]

    top = (
            ratings.
            join(top_users, rsuffix='_r', how='inner', on='userId').
            join(top_movies, rsuffix='_r', how='inner', on='movieId'))
    
    return pd.crosstab(top.userId, top.movieId, top.rating,aggfunc=np.sum)

tabular_preview(ratings, 10)

C:\Users\Admin\AppData\Local\Temp\ipykernel_8624\1087222066.py:14: FutureWarning: The provided callable <function sum at 0x000001DFFEEC9080> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  return pd.crosstab(top.userId, top.movieId, top.rating,aggfunc=np.sum)


movieId,110,260,296,318,356,480,527,589,593,2571
userId,,,,,,,,,,
68,2.5,5.0,2.0,3.0,3.5,3.5,4.0,3.5,3.5,4.5
274,4.5,3.0,5.0,4.5,4.5,3.5,4.0,4.5,4.0,4.0
288,5.0,5.0,5.0,5.0,5.0,2.0,5.0,4.0,5.0,3.0
380,4.0,5.0,5.0,3.0,5.0,5.0,NaN,5.0,5.0,4.5
414,5.0,5.0,5.0,5.0,5.0,4.0,4.0,5.0,4.0,5.0
448,NaN,5.0,5.0,NaN,3.0,3.0,NaN,3.0,5.0,2.0
474,3.0,4.0,4.0,5.0,3.0,4.5,5.0,4.0,4.5,4.5
599,3.5,5.0,5.0,4.0,3.5,4.0,NaN,4.5,3.0,5.0
606,3.5,4.5,5.0,3.5,4.0,2.5,5.0,3.5,4.5,5.0


In [39]:
def create_dataset(ratings, top=None):
    if top is not None:
        ratings.groupby('userId')['ratings'].count()

    unique_users = ratings.userId.unique()
    user_to_index = {old: new for new, old in enumerate(unique_users)}
    new_users = ratings.userId.map(user_to_index)

    unique_movies = ratings.movieId.unique()
    movie_to_index = {old: new for new, old in enumerate(unique_movies)}
    new_movies = ratings.userId.map(movie_to_index)

    n_users = unique_users.shape[0]
    n_movies = unique_movies.shape[0]

    X = pd.DataFrame({'user_id': new_users, 'movie_id': new_movies})
    y = ratings['rating'].astype(np.float32)
    return (n_users, n_movies), (X, y), (user_to_index, movie_to_index)

(n, m), (X, y), (user_to_index, movie_to_index) = create_dataset(ratings)
print(f'Embeddings: {n} users, {m} movies')
print(f'Dataset shape: {X.shape}')
print(f'Target shape: {y.shape}')
    

Embeddings: 610 users, 9724 movies
Dataset shape: (100836, 2)
Target shape: (100836,)


In [40]:
class ReviewsIterator:
    
    def __init__(self, X, y, batch_size=32, shuffle=True):
        X, y = np.asarray(X), np.asarray(y)
        
        if shuffle:
            index = np.random.permutation(X.shape[0])
            X, y = X[index], y[index]
            
        self.X = X
        self.y = y
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n_batches = int(math.ceil(X.shape[0] // batch_size))
        self._current = 0
        
    def __iter__(self):
        return self
    
    def __next__(self):
        return self.next()
    
    def next(self):
        if self._current >= self.n_batches:
            raise StopIteration()
        k = self._current
        self._current += 1
        bs = self.batch_size
        return self.X[k*bs:(k + 1)*bs], self.y[k*bs:(k + 1)*bs]

In [41]:
def batches(X, y, bs=32, shuffle=True):
    for xb, yb in ReviewsIterator(X, y, bs, shuffle):
        xb = torch.LongTensor(xb)
        yb = torch.FloatTensor(yb)
        yield xb, yb.view(-1, 1) 

In [42]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
datasets = {'train': (X_train, y_train), 'val': (X_valid, y_valid)}
dataset_sizes = {'train': len(X_train), 'val': len(X_valid)}

In [43]:
class EmbeddingNet(nn.Module):
    """
    Creates a dense network with embedding layers.
    
    Args:
    
        n_users:            
            Number of unique users in the dataset.

        n_movies: 
            Number of unique movies in the dataset.

        n_factors: 
            Number of columns in the embeddings matrix.

        embedding_dropout: 
            Dropout rate to apply right after embeddings layer.

        hidden:
            A single integer or a list of integers defining the number of 
            units in hidden layer(s).

        dropouts: 
            A single integer or a list of integers defining the dropout 
            layers rates applyied right after each of hidden layers.
            
    """
    def __init__(self, n_users, n_movies,
                 n_factors=50, embedding_dropout=0.02, 
                 hidden=10, dropouts=0.2):
        super().__init__()
        hidden = get_list(hidden)
        dropouts = get_list(dropouts)
        n_last = hidden[-1]
        
        def gen_layers(n_in):
            """
            A generator that yields a sequence of hidden layers and 
            their activations/dropouts.
            
            Note that the function captures `hidden` and `dropouts` 
            values from the outer scope.
            """
            nonlocal hidden, dropouts
            assert len(dropouts) <= len(hidden)
            
            for n_out, rate in zip_longest(hidden, dropouts):
                yield nn.Linear(n_in, n_out)
                yield nn.ReLU()
                if rate is not None and rate > 0.:
                    yield nn.Dropout(rate)
                n_in = n_out
            
        self.u = nn.Embedding(n_users, n_factors)
        self.m = nn.Embedding(n_movies, n_factors)
        self.drop = nn.Dropout(embedding_dropout)
        self.hidden = nn.Sequential(*list(gen_layers(n_factors * 2)))
        self.fc = nn.Linear(n_last, 1)
        self._init()
        
    def forward(self, users, movies, minmax=None):
        features = torch.cat([self.u(users), self.m(movies)], dim=1)
        x = self.drop(features)
        x = self.hidden(x)
        out = torch.sigmoid(self.fc(x))
        if minmax is not None:
            min_rating, max_rating = minmax
            out = out*(max_rating - min_rating + 1) + min_rating - 0.5
        return out
    
    def _init(self):
        """
        Setup embeddings and hidden layers with reasonable initial values.
        """
        def init(m):
            if type(m) == nn.Linear:
                torch.nn.init.xavier_uniform_(m.weight)
                m.bias.data.fill_(0.01)
                
        self.u.weight.data.uniform_(-0.05, 0.05)
        self.m.weight.data.uniform_(-0.05, 0.05)
        self.hidden.apply(init)
        init(self.fc)
    
    
def get_list(n):
    if isinstance(n, (int, float)):
        return [n]
    elif hasattr(n, '__iter__'):
        return list(n)
    raise TypeError('layers configuraiton should be a single number or a list of numbers')

In [44]:
net = EmbeddingNet(
    n_users=n, n_movies=m, 
    n_factors=150, hidden=[500, 500, 500], 
    embedding_dropout=0.05, dropouts=[0.5, 0.5, 0.25])

In [45]:
lr = 1e-5
wd = 1e-5
bs = 200 
n_epochs = 200
patience = 10
no_improvements = 0
best_loss = np.inf
best_weights = None
history = []

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

net.to(device)
criterion = nn.MSELoss(reduction='sum')
optimizer = optim.Adam(net.parameters(), lr=lr, weight_decay=wd)
iterations_per_epoch = int(math.ceil(dataset_sizes['train'] // bs))

for epoch in range(n_epochs):
    stats = {'epoch': epoch + 1, 'total': n_epochs}
    
    for phase in ('train', 'val'):
        training = phase == 'train'
        running_loss = 0.0
        n_batches = 0
        batch_num = 0
        for batch in batches(*datasets[phase], shuffle=training, bs=bs):
            x_batch, y_batch = [b.to(device) for b in batch]
            optimizer.zero_grad()
            # compute gradients only during 'train' phase
            with torch.set_grad_enabled(training):
                outputs = net(x_batch[:, 0], x_batch[:, 1], minmax)
                loss = criterion(outputs, y_batch)
                
                # don't update weights and rates when in 'val' phase
                if training:
                    loss.backward()
                    optimizer.step()
                    
            running_loss += loss.item()
            
        epoch_loss = running_loss / dataset_sizes[phase]
        stats[phase] = epoch_loss
        
        # early stopping: save weights of the best model so far
        if phase == 'val':
            if epoch_loss < best_loss:
                print('loss improvement on epoch: %d' % (epoch + 1))
                best_loss = epoch_loss
                best_weights = copy.deepcopy(net.state_dict())
                no_improvements = 0
            else:
                no_improvements += 1
                
    history.append(stats)
    print('[{epoch:03d}/{total:03d}] train: {train:.4f} - val: {val:.4f}'.format(**stats))
    if no_improvements >= patience:
        print('early stopping after epoch {epoch:03d}'.format(**stats))
        break

loss improvement on epoch: 1
[001/200] train: 1.1863 - val: 1.0836
loss improvement on epoch: 2
[002/200] train: 1.0589 - val: 1.0559
loss improvement on epoch: 3
[003/200] train: 1.0316 - val: 1.0280
loss improvement on epoch: 4
[004/200] train: 0.9969 - val: 0.9940
loss improvement on epoch: 5
[005/200] train: 0.9622 - val: 0.9628
loss improvement on epoch: 6
[006/200] train: 0.9394 - val: 0.9444
loss improvement on epoch: 7
[007/200] train: 0.9256 - val: 0.9385
loss improvement on epoch: 8
[008/200] train: 0.9171 - val: 0.9312
loss improvement on epoch: 9
[009/200] train: 0.9097 - val: 0.9257
loss improvement on epoch: 10
[010/200] train: 0.9070 - val: 0.9251
loss improvement on epoch: 11
[011/200] train: 0.9035 - val: 0.9246
loss improvement on epoch: 12
[012/200] train: 0.9022 - val: 0.9241
loss improvement on epoch: 13
[013/200] train: 0.8979 - val: 0.9176
[014/200] train: 0.8976 - val: 0.9199
loss improvement on epoch: 15
[015/200] train: 0.8930 - val: 0.9148
[016/200] train: 0.